Stage 4 — Preprocessing, Scaling and Train/Test Split
Before code, let me explain what happens in this stage and why each step exists.

What is Preprocessing?
Even though our data is clean and numeric, ML algorithms are sensitive to scale differences between features.
Look at our features:
RecencyDays   : ranges 0 to 571   ← large numbers
RecencyScore  : ranges 0 to 2     ← tiny numbers
Country_USA   : ranges 0 to 1     ← binary
NumInvoices   : ranges 6 to 7     ← very tight
If we feed this directly into a model like Linear Regression:
RecencyDays dominates because its numbers are large
RecencyScore gets ignored because its numbers are tiny
The model thinks RecencyDays is 285x more important
just because of scale — not because of actual signal

What is Scaling?
Scaling brings all features to the same numeric range so the model judges them fairly by signal, not by magnitude.
Two main approaches:
StandardScaler — Z-score normalization:
scaled = (value - mean) / std

Result: mean=0, std=1 for every column
RecencyDays 208 → 0.0  (it was average)
RecencyDays 571 → 2.4  (it was high)
RecencyDays   0 → -2.1 (it was low)

Best for: Linear Regression, Ridge, Lasso, SVM
Assumes roughly normal distribution
MinMaxScaler:
scaled = (value - min) / (max - min)

Result: every value between 0 and 1
RecencyDays   0 → 0.0
RecencyDays 571 → 1.0
RecencyDays 208 → 0.36

Best for: Neural networks, KNN
Sensitive to outliers
We will use StandardScaler — it works best for regression problems and is robust to the slight skew in our data.

What is Train/Test Split?
We cannot train and test a model on the same data. That would be like giving students the exam questions in advance — they score 100% but learned nothing.
All data (59 rows)
      ↓
      ├── Training set (80% = 47 rows)  ← model learns from this
      └── Test set     (20% = 12 rows)  ← model evaluated on this
                                           model never sees this during training
With only 59 customers, we use 80/20 split — a 70/30 split would leave only 17 rows for training which is too few.

What is Cross Validation?
Even 47 training rows is small. Cross validation squeezes more signal from limited data:
Split training data into 5 equal folds:

Fold 1: [test] [train] [train] [train] [train]
Fold 2: [train] [test] [train] [train] [train]
Fold 3: [train] [train] [test] [train] [train]
Fold 4: [train] [train] [train] [test] [train]
Fold 5: [train] [train] [train] [train] [test]

Train 5 times, test 5 times
Average the 5 scores → reliable performance estimate
This gives us a much more honest picture of model performance than a single train/test split.

What is a sklearn Pipeline?
Instead of applying scaling and then training separately, sklearn lets us chain them:
pythonPipeline([
    ("scaler", StandardScaler()),
    ("model",  LinearRegression())
])
Benefits:
1. Scaler is fit ONLY on training data
   then applied to test data
   prevents data leakage from scaling

2. One object to save, load and deploy
   pipeline.predict(X_new) automatically
   scales then predicts in one call

3. Clean, reproducible, production-ready

Which columns need scaling?
Not all columns need scaling equally:
SCALE these:
  NumInvoices   (6-7)
  RecencyDays   (0-571)
  RecencyScore  (0-2)

DO NOT SCALE these (already 0/1):
  All OHE columns  (Country_*, Genre_*, SupportRepId_*)
  Binary columns are already on same scale
We handle this with sklearn's ColumnTransformer — applies different transformations to different columns.

In [27]:
# Cell 1: Imports and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

# sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline

data_path = os.path.join("..", "data")

# Load features_final.csv built in Stage 3
df = pd.read_csv(os.path.join(data_path, "features_final.csv"))

# Load feature metadata
with open(os.path.join(data_path, "feature_meta.json"), "r") as f:
    meta = json.load(f)

print("✓ Data loaded")
print(f"  Shape          : {df.shape}")
print(f"  Feature cols   : {meta['n_features']}")
print(f"  Target col     : {meta['target_col']}")
print()
print("Columns in file:")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}  {col}")

✓ Data loaded
  Shape          : (1000, 38)
  Feature cols   : 34
  Target col     : TotalSpent_log

Columns in file:
   0  CustomerId
   1  NumInvoices
   2  RecencyDays
   3  RecencyScore
   4  Country_grouped_Belgium
   5  Country_grouped_Brazil
   6  Country_grouped_Canada
   7  Country_grouped_Chile
   8  Country_grouped_Czech Republic
   9  Country_grouped_Denmark
  10  Country_grouped_Finland
  11  Country_grouped_France
  12  Country_grouped_Germany
  13  Country_grouped_Hungary
  14  Country_grouped_India
  15  Country_grouped_Ireland
  16  Country_grouped_Italy
  17  Country_grouped_Netherlands
  18  Country_grouped_Norway
  19  Country_grouped_Other
  20  Country_grouped_Portugal
  21  Country_grouped_USA
  22  Country_grouped_United Kingdom
  23  FavoriteGenre_Blues
  24  FavoriteGenre_Classical
  25  FavoriteGenre_Easy Listening
  26  FavoriteGenre_Hip Hop/Rap
  27  FavoriteGenre_Jazz
  28  FavoriteGenre_Latin
  29  FavoriteGenre_Metal
  30  FavoriteGenre_Pop
  31  Favorit

In [28]:
# Cell 2: Separate features X and target y

# Load feature column names from metadata
# This way we never hardcode column names
feature_cols = meta["feature_cols"]
target_col   = meta["target_col"]

# Separate X and y
X = df[feature_cols].copy()
y = df[target_col].copy()

# Keep CustomerId and TotalSpent separate for reference later
customer_ids  = df["CustomerId"].values
total_spent   = df["TotalSpent"].values

print("✓ X and y separated")
print(f"  X shape : {X.shape}")
print(f"  y shape : {y.shape}")
print()
print(f"  Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"    {col}")
print()
print(f"  Target : {target_col}")
print(f"    Min  : {y.min():.4f}")
print(f"    Max  : {y.max():.4f}")
print(f"    Mean : {y.mean():.4f}")

✓ X and y separated
  X shape : (1000, 34)
  y shape : (1000,)

  Feature columns (34):
    RecencyDays
    RecencyScore
    Country_grouped_Belgium
    Country_grouped_Brazil
    Country_grouped_Canada
    Country_grouped_Chile
    Country_grouped_Czech Republic
    Country_grouped_Denmark
    Country_grouped_Finland
    Country_grouped_France
    Country_grouped_Germany
    Country_grouped_Hungary
    Country_grouped_India
    Country_grouped_Ireland
    Country_grouped_Italy
    Country_grouped_Netherlands
    Country_grouped_Norway
    Country_grouped_Other
    Country_grouped_Portugal
    Country_grouped_USA
    Country_grouped_United Kingdom
    FavoriteGenre_Blues
    FavoriteGenre_Classical
    FavoriteGenre_Easy Listening
    FavoriteGenre_Hip Hop/Rap
    FavoriteGenre_Jazz
    FavoriteGenre_Latin
    FavoriteGenre_Metal
    FavoriteGenre_Pop
    FavoriteGenre_R&B/Soul
    FavoriteGenre_Reggae
    FavoriteGenre_Rock
    SupportRepId_4
    SupportRepId_5

  Target : TotalSpent_

**Import MLFlow and Logging all Parameters / Metrics **

In [29]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")  # if using MLflow server
mlflow.set_experiment("Chinook_End_to_End_Project")

with mlflow.start_run(run_name="Model Training"):
    mlflow.log_param("stage", "preprocessing and training")

    mlflow.log_param("total_rows", len(df))
    mlflow.log_param("total_columns", df.shape[1])

    mlflow.log_param("target_column", meta['target_col'])

    mlflow.log_param("initial_feature_count", meta["n_features"])

🏃 View run Model Training at: http://127.0.0.1:5000/#/experiments/970861043057358249/runs/e2a085a2975246eaae4eb4d10b59bd09
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/970861043057358249


In [30]:
# Cell 3: Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,     # 20% for testing = ~12 rows
    random_state = 42,       # fixed seed → reproducible split every run
    shuffle      = True,      # shuffle before splitting
    stratify     = X["RecencyScore"]  # ensure balanced recency groups
)

# Also split customer_ids and total_spent to keep them aligned
ids_train, ids_test         = train_test_split(
    customer_ids, test_size=0.20, random_state=42, shuffle=True,  stratify     = X["RecencyScore"]  # ensure balanced recency groups
)
spent_train, spent_test     = train_test_split(
    total_spent, test_size=0.20, random_state=42, shuffle=True,  stratify     = X["RecencyScore"]  # ensure balanced recency groups
)   

print("✓ Train/Test split complete")
print()
print(f"  Total rows   : {len(X)}")
print(f"  Training set : {X_train.shape[0]} rows "
      f"({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Test set     : {X_test.shape[0]} rows  "
      f"({X_test.shape[0]/len(X)*100:.1f}%)")
print()

# Verify target distribution is similar in both splits
# If train has all high spenders and test has all low spenders
# the model will be evaluated unfairly
print("Target distribution check:")
print(f"  {'Metric':<10} {'Train':>10} {'Test':>10} {'Full':>10}")
print(f"  {'-'*40}")
print(f"  {'Min':<10} {y_train.min():>10.4f} "
      f"{y_test.min():>10.4f} {y.min():>10.4f}")
print(f"  {'Max':<10} {y_train.max():>10.4f} "
      f"{y_test.max():>10.4f} {y.max():>10.4f}")
print(f"  {'Mean':<10} {y_train.mean():>10.4f} "
      f"{y_test.mean():>10.4f} {y.mean():>10.4f}")
print(f"  {'Std':<10} {y_train.std():>10.4f} "
      f"{y_test.std():>10.4f} {y.std():>10.4f}")
print()

# Verify class balance for categorical features
print("Country distribution check (train vs test):")
train_df = X_train.copy()
test_df  = X_test.copy()

country_cols = [c for c in feature_cols
                if c.startswith("Country")]
print(f"  {'Country':<30} {'Train %':>10} {'Test %':>10}")
print(f"  {'-'*52}")
for col in country_cols:
    train_pct = train_df[col].mean() * 100
    test_pct  = test_df[col].mean()  * 100
    print(f"  {col:<30} {train_pct:>9.1f}% {test_pct:>9.1f}%")

✓ Train/Test split complete

  Total rows   : 1000
  Training set : 800 rows (80.0%)
  Test set     : 200 rows  (20.0%)

Target distribution check:
  Metric          Train       Test       Full
  ----------------------------------------
  Min            0.6366     0.6627     0.6366
  Max            4.5406     4.4049     4.5406
  Mean           2.8637     2.9720     2.8854
  Std            0.8558     0.8364     0.8526

Country distribution check (train vs test):
  Country                           Train %     Test %
  ----------------------------------------------------
  Country_grouped_Belgium              1.5%       5.0%
  Country_grouped_Brazil               8.0%       7.0%
  Country_grouped_Canada              13.9%       9.5%
  Country_grouped_Chile                2.2%       0.0%
  Country_grouped_Czech Republic       4.6%       3.5%
  Country_grouped_Denmark              2.1%       0.5%
  Country_grouped_Finland              2.2%       0.5%
  Country_grouped_France               

In [31]:
mlflow.log_param("test_size", 0.20)
mlflow.log_param("random_state", 42)
mlflow.log_param("shuffle", True)
mlflow.log_param("stratify_column", "RecencyScore")
mlflow.log_metric("train_rows", X_train.shape[0])
mlflow.log_metric("test_rows", X_test.shape[0])

mlflow.log_metric("train_percentage",
                  X_train.shape[0]/len(X)*100)

mlflow.log_metric("test_percentage",
                  X_test.shape[0]/len(X)*100)
mlflow.log_metric("y_train_mean", float(y_train.mean()))
mlflow.log_metric("y_test_mean", float(y_test.mean()))

mlflow.log_metric("y_train_std", float(y_train.std()))
mlflow.log_metric("y_test_std", float(y_test.std()))

mlflow.log_metric("y_train_min", float(y_train.min()))
mlflow.log_metric("y_train_max", float(y_train.max()))

mlflow.log_metric("y_test_min", float(y_test.min()))
mlflow.log_metric("y_test_max", float(y_test.max()))

In [32]:
print(X.head())

   RecencyDays  RecencyScore  Country_grouped_Belgium  Country_grouped_Brazil  \
0         1325             2                        0                       0   
1         1508             2                        0                       0   
2          243             0                        0                       0   
3         1280             2                        0                       0   
4          178             0                        1                       0   

   Country_grouped_Canada  Country_grouped_Chile  \
0                       0                      0   
1                       0                      0   
2                       0                      0   
3                       0                      0   
4                       0                      0   

   Country_grouped_Czech Republic  Country_grouped_Denmark  \
0                               0                        0   
1                               0                        0   
2             

In [33]:
print(meta["feature_cols"])
print("NumInvoices" in meta["feature_cols"])

print(X.columns.tolist())
print("NumInvoices" in X.columns)

['RecencyDays', 'RecencyScore', 'Country_grouped_Belgium', 'Country_grouped_Brazil', 'Country_grouped_Canada', 'Country_grouped_Chile', 'Country_grouped_Czech Republic', 'Country_grouped_Denmark', 'Country_grouped_Finland', 'Country_grouped_France', 'Country_grouped_Germany', 'Country_grouped_Hungary', 'Country_grouped_India', 'Country_grouped_Ireland', 'Country_grouped_Italy', 'Country_grouped_Netherlands', 'Country_grouped_Norway', 'Country_grouped_Other', 'Country_grouped_Portugal', 'Country_grouped_USA', 'Country_grouped_United Kingdom', 'FavoriteGenre_Blues', 'FavoriteGenre_Classical', 'FavoriteGenre_Easy Listening', 'FavoriteGenre_Hip Hop/Rap', 'FavoriteGenre_Jazz', 'FavoriteGenre_Latin', 'FavoriteGenre_Metal', 'FavoriteGenre_Pop', 'FavoriteGenre_R&B/Soul', 'FavoriteGenre_Reggae', 'FavoriteGenre_Rock', 'SupportRepId_4', 'SupportRepId_5']
False
['RecencyDays', 'RecencyScore', 'Country_grouped_Belgium', 'Country_grouped_Brazil', 'Country_grouped_Canada', 'Country_grouped_Chile', 'C

In [34]:
# Cell 4: Define which columns need scaling and which don't

print("=== Column Groups for Preprocessing ===")
print()

# -------------------------------------------------------
# Group 1: Numeric columns — need StandardScaler
# These have different ranges so scaling is essential
# -------------------------------------------------------
numeric_cols = [
    "RecencyDays",    # range 0-571
    "RecencyScore",   # range 0-2
]

# -------------------------------------------------------
# Group 2: Binary/OHE columns — NO scaling needed
# Already on same scale (0 or 1)
# Scaling would distort their binary meaning
# -------------------------------------------------------
binary_cols = [c for c in feature_cols
               if c not in numeric_cols]

print(f"Numeric columns (will be scaled):")
for col in numeric_cols:
    print(f"  {col:<30} "
          f"range: {X[col].min():.1f} to {X[col].max():.1f}")

print()
print(f"Binary/OHE columns (no scaling):")
for col in binary_cols:
    print(f"  {col:<30} "
          f"values: {sorted(X[col].unique())}")

print()

# -------------------------------------------------------
# Verify all feature columns are accounted for
# -------------------------------------------------------
all_accounted = set(numeric_cols + binary_cols) == set(feature_cols)
print(f"All features accounted for : {all_accounted} ✓"
      if all_accounted else
      f"⚠ Missing columns: "
      f"{set(feature_cols) - set(numeric_cols + binary_cols)}")

=== Column Groups for Preprocessing ===

Numeric columns (will be scaled):
  RecencyDays                    range: 2.0 to 1826.0
  RecencyScore                   range: 0.0 to 2.0

Binary/OHE columns (no scaling):
  Country_grouped_Belgium        values: [np.int64(0), np.int64(1)]
  Country_grouped_Brazil         values: [np.int64(0), np.int64(1)]
  Country_grouped_Canada         values: [np.int64(0), np.int64(1)]
  Country_grouped_Chile          values: [np.int64(0), np.int64(1)]
  Country_grouped_Czech Republic values: [np.int64(0), np.int64(1)]
  Country_grouped_Denmark        values: [np.int64(0), np.int64(1)]
  Country_grouped_Finland        values: [np.int64(0), np.int64(1)]
  Country_grouped_France         values: [np.int64(0), np.int64(1)]
  Country_grouped_Germany        values: [np.int64(0), np.int64(1)]
  Country_grouped_Hungary        values: [np.int64(0), np.int64(1)]
  Country_grouped_India          values: [np.int64(0), np.int64(1)]
  Country_grouped_Ireland        value

In [35]:
# Cell 5: Build ColumnTransformer and Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.compose       import ColumnTransformer
from sklearn.pipeline      import Pipeline
from sklearn.linear_model  import LinearRegression

print("=== Building Preprocessing Pipeline ===")
print()

# -------------------------------------------------------
# Step 1: ColumnTransformer
# Applies different transformations to different columns
# numeric_cols → StandardScaler
# binary_cols  → passthrough (no transformation)
# -------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("scale",       # name — can be anything
         StandardScaler(),
         numeric_cols),  # apply to these columns

        ("passthrough",  # name
         "passthrough",  # keyword → do nothing, just pass through
         binary_cols),   # apply to these columns
    ]
)

print("ColumnTransformer defined:")
print(f"  scale       → StandardScaler on {numeric_cols}")
print(f"  passthrough → no scaling on {len(binary_cols)} binary columns")
print()

# -------------------------------------------------------
# Step 2: Fit preprocessor on TRAINING data only
# CRITICAL: never fit on test data
# Fitting on test data = data leakage
# We learn mean and std from train, apply to both
# -------------------------------------------------------
preprocessor.fit(X_train)

print("✓ Preprocessor fitted on training data only")
print()

# -------------------------------------------------------
# Step 3: Transform both train and test
# -------------------------------------------------------
X_train_scaled = preprocessor.transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

print(f"✓ Transformation applied")
print(f"  X_train_scaled shape : {X_train_scaled.shape}")
print(f"  X_test_scaled  shape : {X_test_scaled.shape}")
print()

# -------------------------------------------------------
# Step 4: Verify scaling worked correctly
# Scaled numeric columns should have mean≈0 and std≈1
# Binary columns should be unchanged (still 0 or 1)
# -------------------------------------------------------
print("=== Scaling Verification ===")
print()

# Convert back to DataFrame for easy inspection
col_order = numeric_cols + binary_cols
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=col_order
)
X_test_scaled_df  = pd.DataFrame(
    X_test_scaled,
    columns=col_order
)

print("Numeric columns after scaling (train):")
print(f"  {'Column':<30} {'Mean':>10} {'Std':>10} "
      f"{'Min':>10} {'Max':>10}")
print(f"  {'-'*62}")
for col in numeric_cols:
    print(f"  {col:<30} "
          f"{X_train_scaled_df[col].mean():>10.4f} "
          f"{X_train_scaled_df[col].std():>10.4f} "
          f"{X_train_scaled_df[col].min():>10.4f} "
          f"{X_train_scaled_df[col].max():>10.4f}")

print()
print("Binary columns after passthrough (train) — should be 0 or 1:")
print(f"  {'Column':<30} {'Min':>10} {'Max':>10} {'Mean':>10}")
print(f"  {'-'*62}")
for col in binary_cols:
    print(f"  {col:<30} "
          f"{X_train_scaled_df[col].min():>10.4f} "
          f"{X_train_scaled_df[col].max():>10.4f} "
          f"{X_train_scaled_df[col].mean():>10.4f}")

=== Building Preprocessing Pipeline ===

ColumnTransformer defined:
  scale       → StandardScaler on ['RecencyDays', 'RecencyScore']
  passthrough → no scaling on 32 binary columns

✓ Preprocessor fitted on training data only

✓ Transformation applied
  X_train_scaled shape : (800, 34)
  X_test_scaled  shape : (200, 34)

=== Scaling Verification ===

Numeric columns after scaling (train):
  Column                               Mean        Std        Min        Max
  --------------------------------------------------------------
  RecencyDays                        0.0000     1.0006    -1.6945     1.7700
  RecencyScore                      -0.0000     1.0006    -1.2340     1.2096

Binary columns after passthrough (train) — should be 0 or 1:
  Column                                Min        Max       Mean
  --------------------------------------------------------------
  Country_grouped_Belgium            0.0000     1.0000     0.0150
  Country_grouped_Brazil             0.0000     1.00

In [36]:
mlflow.log_param(
    "numeric_columns",
    ",".join(numeric_cols)
)

mlflow.log_param(
    "binary_columns_count",
    len(binary_cols)
)

mlflow.log_param(
    "scaler",
    "StandardScaler"
)

mlflow.log_param(
    "preprocessor",
    "ColumnTransformer"
)

'ColumnTransformer'

In [39]:
# Diagnostic: check NumInvoices in training set
# print("NumInvoices distribution in training set:")
# print(X_train["NumInvoices"].value_counts().sort_index())
# print(f"Std in training set: {X_train['NumInvoices'].std():.6f}")
print()
# print("This confirms NumInvoices has near-zero variance")
# print("in the training set — it adds no signal")
# print("StandardScaler divided by ~0 → all values become 0")

Doing again train/test split due to Numinvoices has zero variances

In [ ]:
# Cell 6: Train/Test Split again

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,     # 20% for testing = ~12 rows
    random_state = 42,       # fixed seed → reproducible split every run
    shuffle      = True,      # shuffle before splitting
    stratify     = X["RecencyScore"]  # ensure balanced recency groups
)

# Also split customer_ids and total_spent to keep them aligned
ids_train, ids_test         = train_test_split(
    customer_ids, test_size=0.20, random_state=42, shuffle=True,  stratify     = X["RecencyScore"]  # ensure balanced recency groups
)
spent_train, spent_test     = train_test_split(
    total_spent, test_size=0.20, random_state=42, shuffle=True,  stratify     = X["RecencyScore"]  # ensure balanced recency groups
)   

# Remove from both X splits
X_train = X_train.drop(columns=["NumInvoices"])
X_test  = X_test.drop(columns=["NumInvoices"])

# Update feature lists
numeric_cols  = ["RecencyDays", "RecencyScore"]
feature_cols  = numeric_cols + binary_cols

# Update metadata
meta["feature_cols"]  = feature_cols
meta["numeric_cols"]  = numeric_cols
meta["n_features"]    = len(feature_cols)


print("✓ Train/Test split complete")
print()
print(f"  Total rows   : {len(X)}")
print(f"  Training set : {X_train.shape[0]} rows "
      f"({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Test set     : {X_test.shape[0]} rows  "
      f"({X_test.shape[0]/len(X)*100:.1f}%)")
print()

# Verify target distribution is similar in both splits
# If train has all high spenders and test has all low spenders
# the model will be evaluated unfairly
print("Target distribution check:")
print(f"  {'Metric':<10} {'Train':>10} {'Test':>10} {'Full':>10}")
print(f"  {'-'*40}")
print(f"  {'Min':<10} {y_train.min():>10.4f} "
      f"{y_test.min():>10.4f} {y.min():>10.4f}")
print(f"  {'Max':<10} {y_train.max():>10.4f} "
      f"{y_test.max():>10.4f} {y.max():>10.4f}")
print(f"  {'Mean':<10} {y_train.mean():>10.4f} "
      f"{y_test.mean():>10.4f} {y.mean():>10.4f}")
print(f"  {'Std':<10} {y_train.std():>10.4f} "
      f"{y_test.std():>10.4f} {y.std():>10.4f}")
print()

# Verify class balance for categorical features
print("Country distribution check (train vs test):")
train_df = X_train.copy()
test_df  = X_test.copy()

country_cols = [c for c in feature_cols
                if c.startswith("Country")]
print(f"  {'Country':<30} {'Train %':>10} {'Test %':>10}")
print(f"  {'-'*52}")
for col in country_cols:
    train_pct = train_df[col].mean() * 100
    test_pct  = test_df[col].mean()  * 100
    print(f"  {col:<30} {train_pct:>9.1f}% {test_pct:>9.1f}%")

✓ Train/Test split complete

  Total rows   : 1000
  Training set : 800 rows (80.0%)
  Test set     : 200 rows  (20.0%)

Target distribution check:
  Metric          Train       Test       Full
  ----------------------------------------
  Min            0.6366     0.6627     0.6366
  Max            4.5406     4.4049     4.5406
  Mean           2.8637     2.9720     2.8854
  Std            0.8558     0.8364     0.8526

Country distribution check (train vs test):
  Country                           Train %     Test %
  ----------------------------------------------------
  Country_grouped_Belgium              1.5%       5.0%
  Country_grouped_Brazil               8.0%       7.0%
  Country_grouped_Canada              13.9%       9.5%
  Country_grouped_Chile                2.2%       0.0%
  Country_grouped_Czech Republic       4.6%       3.5%
  Country_grouped_Denmark              2.1%       0.5%
  Country_grouped_Finland              2.2%       0.5%
  Country_grouped_France               

In [ ]:
mlflow.log_param(
    "dropped_zero_variance_feature",
    "NumInvoices"
)
mlflow.log_param(
    "dropped_features",
    "NumInvoices,TenureDays"
)

In [ ]:
# Cell 7: Define which columns need scaling and which don't

print("=== Column Groups for Preprocessing ===")
print()

# -------------------------------------------------------
# Group 1: Numeric columns — need StandardScaler
# These have different ranges so scaling is essential
# -------------------------------------------------------
numeric_cols = [
    "RecencyDays",    # range 0-571
    "RecencyScore",   # range 0-2
]

# -------------------------------------------------------
# Group 2: Binary/OHE columns — NO scaling needed
# Already on same scale (0 or 1)
# Scaling would distort their binary meaning
# -------------------------------------------------------
binary_cols = [c for c in feature_cols
               if c not in numeric_cols]

print(f"Numeric columns (will be scaled):")
for col in numeric_cols:
    print(f"  {col:<30} "
          f"range: {X[col].min():.1f} to {X[col].max():.1f}")

print()
print(f"Binary/OHE columns (no scaling):")
for col in binary_cols:
    print(f"  {col:<30} "
          f"values: {sorted(X[col].unique())}")

print()

# -------------------------------------------------------
# Verify all feature columns are accounted for
# -------------------------------------------------------
all_accounted = set(numeric_cols + binary_cols) == set(feature_cols)
print(f"All features accounted for : {all_accounted} ✓"
      if all_accounted else
      f"⚠ Missing columns: "
      f"{set(feature_cols) - set(numeric_cols + binary_cols)}")

=== Column Groups for Preprocessing ===

Numeric columns (will be scaled):
  RecencyDays                    range: 2.0 to 1826.0
  RecencyScore                   range: 0.0 to 2.0

Binary/OHE columns (no scaling):
  Country_grouped_Belgium        values: [np.int64(0), np.int64(1)]
  Country_grouped_Brazil         values: [np.int64(0), np.int64(1)]
  Country_grouped_Canada         values: [np.int64(0), np.int64(1)]
  Country_grouped_Chile          values: [np.int64(0), np.int64(1)]
  Country_grouped_Czech Republic values: [np.int64(0), np.int64(1)]
  Country_grouped_Denmark        values: [np.int64(0), np.int64(1)]
  Country_grouped_Finland        values: [np.int64(0), np.int64(1)]
  Country_grouped_France         values: [np.int64(0), np.int64(1)]
  Country_grouped_Germany        values: [np.int64(0), np.int64(1)]
  Country_grouped_Hungary        values: [np.int64(0), np.int64(1)]
  Country_grouped_India          values: [np.int64(0), np.int64(1)]
  Country_grouped_Ireland        value

In [ ]:
# Cell 8: Build ColumnTransformer and Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.compose       import ColumnTransformer
from sklearn.pipeline      import Pipeline
from sklearn.linear_model  import LinearRegression

print("=== Building Preprocessing Pipeline ===")
print()

# -------------------------------------------------------
# Step 1: ColumnTransformer
# Applies different transformations to different columns
# numeric_cols → StandardScaler
# binary_cols  → passthrough (no transformation)
# -------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("scale",       # name — can be anything
         StandardScaler(),
         numeric_cols),  # apply to these columns

        ("passthrough",  # name
         "passthrough",  # keyword → do nothing, just pass through
         binary_cols),   # apply to these columns
    ]
)

print("ColumnTransformer defined:")
print(f"  scale       → StandardScaler on {numeric_cols}")
print(f"  passthrough → no scaling on {len(binary_cols)} binary columns")
print()

# -------------------------------------------------------
# Step 2: Fit preprocessor on TRAINING data only
# CRITICAL: never fit on test data
# Fitting on test data = data leakage
# We learn mean and std from train, apply to both
# -------------------------------------------------------
preprocessor.fit(X_train)

print("✓ Preprocessor fitted on training data only")
print()

# -------------------------------------------------------
# Step 3: Transform both train and test
# -------------------------------------------------------
X_train_scaled = preprocessor.transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

print(f"✓ Transformation applied")
print(f"  X_train_scaled shape : {X_train_scaled.shape}")
print(f"  X_test_scaled  shape : {X_test_scaled.shape}")
print()

# -------------------------------------------------------
# Step 4: Verify scaling worked correctly
# Scaled numeric columns should have mean≈0 and std≈1
# Binary columns should be unchanged (still 0 or 1)
# -------------------------------------------------------
print("=== Scaling Verification ===")
print()

# Convert back to DataFrame for easy inspection
col_order = numeric_cols + binary_cols
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=col_order
)
X_test_scaled_df  = pd.DataFrame(
    X_test_scaled,
    columns=col_order
)

print("Numeric columns after scaling (train):")
print(f"  {'Column':<30} {'Mean':>10} {'Std':>10} "
      f"{'Min':>10} {'Max':>10}")
print(f"  {'-'*62}")
for col in numeric_cols:
    print(f"  {col:<30} "
          f"{X_train_scaled_df[col].mean():>10.4f} "
          f"{X_train_scaled_df[col].std():>10.4f} "
          f"{X_train_scaled_df[col].min():>10.4f} "
          f"{X_train_scaled_df[col].max():>10.4f}")

print()
print("Binary columns after passthrough (train) — should be 0 or 1:")
print(f"  {'Column':<30} {'Min':>10} {'Max':>10} {'Mean':>10}")
print(f"  {'-'*62}")
for col in binary_cols:
    print(f"  {col:<30} "
          f"{X_train_scaled_df[col].min():>10.4f} "
          f"{X_train_scaled_df[col].max():>10.4f} "
          f"{X_train_scaled_df[col].mean():>10.4f}")

=== Building Preprocessing Pipeline ===

ColumnTransformer defined:
  scale       → StandardScaler on ['RecencyDays', 'RecencyScore']
  passthrough → no scaling on 32 binary columns

✓ Preprocessor fitted on training data only

✓ Transformation applied
  X_train_scaled shape : (800, 34)
  X_test_scaled  shape : (200, 34)

=== Scaling Verification ===

Numeric columns after scaling (train):
  Column                               Mean        Std        Min        Max
  --------------------------------------------------------------
  RecencyDays                        0.0000     1.0006    -1.6945     1.7700
  RecencyScore                      -0.0000     1.0006    -1.2340     1.2096

Binary columns after passthrough (train) — should be 0 or 1:
  Column                                Min        Max       Mean
  --------------------------------------------------------------
  Country_grouped_Belgium            0.0000     1.0000     0.0150
  Country_grouped_Brazil             0.0000     1.00

In [ ]:
# Cell 6: Save preprocessor and scaled arrays

import joblib
import os
import json
import numpy as np

models_path = os.path.join("..", "models/data")
os.makedirs(models_path, exist_ok=True)

# -------------------------------------------------------
# Save 1: Preprocessor (fitted ColumnTransformer)
# This is the fitted scaler — must be saved so we
# apply identical scaling when serving predictions
# in production. Never refit on new data.
# -------------------------------------------------------
joblib.dump(
    preprocessor,
    os.path.join(models_path, "preprocessor.joblib")
)
print("✓ Saved preprocessor.joblib")

# -------------------------------------------------------
# Save 2: Scaled arrays as numpy files
# These are the ready-to-train arrays for Stage 5
# -------------------------------------------------------
np.save(os.path.join(models_path, "X_train.npy"), X_train_scaled)
np.save(os.path.join(models_path, "X_test.npy"),  X_test_scaled)
np.save(os.path.join(models_path, "y_train.npy"), y_train.values)
np.save(os.path.join(models_path, "y_test.npy"),  y_test.values)

# Save customer ids and raw spend for result interpretation
np.save(os.path.join(models_path, "ids_train.npy"),   ids_train)
np.save(os.path.join(models_path, "ids_test.npy"),    ids_test)
np.save(os.path.join(models_path, "spent_train.npy"), spent_train)
np.save(os.path.join(models_path, "spent_test.npy"),  spent_test)

print("✓ Saved scaled arrays:")
print(f"  X_train.npy    : {X_train_scaled.shape}")
print(f"  X_test.npy     : {X_test_scaled.shape}")
print(f"  y_train.npy    : {y_train.shape}")
print(f"  y_test.npy     : {y_test.shape}")
print()

# -------------------------------------------------------
# Save 3: Update feature metadata with final info
# -------------------------------------------------------
meta["numeric_cols"]     = numeric_cols
meta["binary_cols"]      = binary_cols
meta["feature_cols"]     = numeric_cols + binary_cols
meta["n_features"]       = len(numeric_cols + binary_cols)
meta["train_size"]       = int(X_train_scaled.shape[0])
meta["test_size"]        = int(X_test_scaled.shape[0])
meta["random_state"]     = 42
meta["dropped_features"]["zero_variance"].append("NumInvoices")

with open(os.path.join(data_path, "feature_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("✓ feature_meta.json updated")
print()

# -------------------------------------------------------
# Stage 4 summary
# -------------------------------------------------------
print("=" * 55)
print("STAGE 4 COMPLETE")
print("=" * 55)
print(f"""
  Preprocessing  : ColumnTransformer
    Numeric cols : StandardScaler → mean=0, std=1
    Binary cols  : passthrough → unchanged

  Train/Test Split:
    Strategy     : stratified on RecencyScore
    Train        : {X_train_scaled.shape[0]} rows (79.7%)
    Test         : {X_test_scaled.shape[0]} rows (20.3%)
    Random state : 42 (reproducible)

  Features       : {len(numeric_cols + binary_cols)} total
    Numeric      : {numeric_cols}
    Binary       : {len(binary_cols)} OHE columns

  Dropped        : NumInvoices (zero variance after split)

  Saved          :
    preprocessor.joblib  ← fitted scaler
    X_train.npy          ← scaled training features
    X_test.npy           ← scaled test features
    y_train.npy          ← training targets
    y_test.npy           ← test targets

  Ready for Stage 5 → Model Training
""")
print("=" * 55)

✓ Saved preprocessor.joblib
✓ Saved scaled arrays:
  X_train.npy    : (800, 34)
  X_test.npy     : (200, 34)
  y_train.npy    : (800,)
  y_test.npy     : (200,)

✓ feature_meta.json updated

STAGE 4 COMPLETE

  Preprocessing  : ColumnTransformer
    Numeric cols : StandardScaler → mean=0, std=1
    Binary cols  : passthrough → unchanged

  Train/Test Split:
    Strategy     : stratified on RecencyScore
    Train        : 800 rows (79.7%)
    Test         : 200 rows (20.3%)
    Random state : 42 (reproducible)

  Features       : 34 total
    Numeric      : ['RecencyDays', 'RecencyScore']
    Binary       : 32 OHE columns

  Dropped        : NumInvoices (zero variance after split)

  Saved          :
    preprocessor.joblib  ← fitted scaler
    X_train.npy          ← scaled training features
    X_test.npy           ← scaled test features
    y_train.npy          ← training targets
    y_test.npy           ← test targets

  Ready for Stage 5 → Model Training



In [ ]:
scaler = preprocessor.named_transformers_["scale"]

for col, mean, scale in zip(
        numeric_cols,
        scaler.mean_,
        scaler.scale_
):

    mlflow.log_metric(
        f"{col}_mean_before_scaling",
        float(mean)
    )

    mlflow.log_metric(
        f"{col}_std_before_scaling",
        float(scale)
    )

mlflow.log_artifact(
    os.path.join(models_path,
                 "preprocessor.joblib")
)

mlflow.log_artifact(
    os.path.join(data_path,
                 "feature_meta.json")
)
mlflow.log_artifact(
    os.path.join(models_path,
                 "X_train.npy")
)

mlflow.log_artifact(
    os.path.join(models_path,
                 "X_test.npy")
)

mlflow.log_artifact(
    os.path.join(models_path,
                 "y_train.npy")
)

mlflow.log_artifact(
    os.path.join(models_path,
                 "y_test.npy")
)

In [ ]:
summary = {
    "train_rows": int(X_train.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "numeric_features": numeric_cols,
    "binary_features": binary_cols,
    "dropped_features": ["NumInvoices"],
    "scaler": "StandardScaler"
}

with open("stage4_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

mlflow.log_artifact("stage4_summary.json")

In [ ]:
mlflow.end_run()